# Notebook 1 — Fire & Smoke Detection (YOLOv8s) — IMPROVED
**Kaggle T4x2 · ~45 phút**

### Thay đổi so với v1:
- `yolov8n` → **`yolov8s`** (backbone lớn hơn, +15% mAP)
- `imgsz 416` → **`640`** (nhận lửa/khói nhỏ xa hơn)
- Thêm **augmentation mạnh**: mosaic, hsv, degrees, scale
- `epochs 100` → **`150`** + `patience 30`
- **Label smoothing 0.1** giảm overfit
- **Hiển thị đầy đủ**: curves, confusion matrix, labels, results, train_batch

In [ ]:
!nvidia-smi
import torch
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
!pip install ultralytics -q

In [ ]:
import os, glob, shutil
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import display
from ultralytics import YOLO

BASE      = Path('/kaggle/input/datasets/nguynteincong/dataset-fall-fire/datasets')
FIRE_ROOT = BASE / 'Fire_and_Smoke'

for split in ['train', 'valid', 'test']:
    imgs = len(list((FIRE_ROOT / split / 'images').glob('*')))
    lbs  = len(list((FIRE_ROOT / split / 'labels').glob('*')))
    print(f'{split:6s} → images: {imgs:4d}  labels: {lbs:4d}')

In [ ]:
yaml_content = f"""path: {FIRE_ROOT}
train: train/images
val:   valid/images
test:  test/images

nc: 2
names: ['fire', 'smoke']
"""
yaml_path = '/kaggle/working/fire_data.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
print(yaml_content)

In [ ]:
model = YOLO('yolov8s.pt')

results = model.train(
    data    = yaml_path,
    epochs  = 150,
    imgsz   = 640,
    batch   = 16,
    device  = '0,1',
    cache    = 'disk',  # disk thay ram: deterministic + tránh warning
    project = '/kaggle/working/runs',
    name     = 'fire_v2',
    exist_ok = True,    # chạy lại cell không bị lỗi tên trùng
    patience = 30,
    save    = True,
    plots   = True,   # <-- tự động sinh toàn bộ biểu đồ

    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 5.0,
    scale        = 0.6,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.1,
    copy_paste   = 0.1,

    # label_smoothing bị deprecated trong ultralytics 8.4+ — đã xóa
    dropout         = 0.0,
    weight_decay    = 0.0005,

    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    warmup_epochs = 5,
)

RUN_DIR = Path('/kaggle/working/runs/fire_v2')
print(f'\nRun dir: {RUN_DIR}')

## Đánh giá trên tập Test

In [ ]:
best_path  = glob.glob(str(RUN_DIR / 'weights/best.pt'))[0]
best_model = YOLO(best_path)

metrics = best_model.val(
    data    = yaml_path,
    split   = 'test',
    imgsz   = 640,
    device  = 0,
    plots   = True,   # sinh confusion matrix trên test set
    save_json = True,
)

print('\n' + '='*50)
print('KẾT QUẢ TEST SET')
print('='*50)
print(f'  mAP50    : {metrics.box.map50:.4f}   (mục tiêu > 0.90)')
print(f'  mAP50-95 : {metrics.box.map:.4f}')
print(f'  Precision: {metrics.box.mp:.4f}')
print(f'  Recall   : {metrics.box.mr:.4f}')
print('='*50)

## Hiển thị toàn bộ kết quả training

In [ ]:
# ── results.csv — bảng metrics theo epoch ─────────────────────────────────────
csv_path = RUN_DIR / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    print('results.csv — 10 epoch cuối:')
    display(df.tail(10).to_string(index=False))
else:
    print('Không tìm thấy results.csv')

In [ ]:
# ── Hàm tiện ích hiển thị ảnh từ run dir ──────────────────────────────────────
def show_image(path, title='', figsize=(12, 6)):
    p = Path(path)
    if not p.exists():
        print(f'  ⚠ Không tìm thấy: {p.name}')
        return
    img = mpimg.imread(str(p))
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title or p.name, fontsize=13, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.show()

def show_images_row(paths, titles=None, figsize=(20, 5)):
    """Hiển thị nhiều ảnh trên cùng 1 hàng."""
    valid = [(p, t) for p, t in zip(paths, titles or paths)
             if Path(p).exists()]
    if not valid:
        print('  ⚠ Không tìm thấy ảnh nào')
        return
    n = len(valid)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, (p, t) in zip(axes, valid):
        ax.imshow(mpimg.imread(p))
        ax.axis('off')
        ax.set_title(t, fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('Hàm hiển thị sẵn sàng.')

In [ ]:
# ── results.png — tổng hợp loss + metrics theo epoch ──────────────────────────
show_image(RUN_DIR / 'results.png', 'Training Results (Loss & Metrics theo Epoch)', figsize=(16, 8))

In [ ]:
# ── Các Curves: F1, Precision, Recall, PR ─────────────────────────────────────
print('=== CURVES ===')
show_images_row(
    [
        RUN_DIR / 'BoxF1_curve.png',
        RUN_DIR / 'BoxP_curve.png',
        RUN_DIR / 'BoxR_curve.png',
        RUN_DIR / 'BoxPR_curve.png',
    ],
    ['F1 Curve', 'Precision Curve', 'Recall Curve', 'PR Curve'],
    figsize=(22, 5)
)

In [ ]:
# ── Confusion Matrix (thường + normalized) ────────────────────────────────────
print('=== CONFUSION MATRIX ===')
show_images_row(
    [
        RUN_DIR / 'confusion_matrix.png',
        RUN_DIR / 'confusion_matrix_normalized.png',
    ],
    ['Confusion Matrix', 'Confusion Matrix (Normalized)'],
    figsize=(16, 6)
)

In [ ]:
# ── Labels distribution ───────────────────────────────────────────────────────
print('=== LABELS DISTRIBUTION ===')
show_image(RUN_DIR / 'labels.jpg', 'Labels Distribution', figsize=(12, 8))

In [ ]:
# ── Train batch mẫu ───────────────────────────────────────────────────────────
print('=== TRAIN BATCH SAMPLES ===')
train_batches = sorted(RUN_DIR.glob('train_batch*.jpg'))[:3]
show_images_row(
    [str(p) for p in train_batches],
    [p.name for p in train_batches],
    figsize=(22, 7)
)

In [ ]:
# ── Val batch mẫu (predictions) ───────────────────────────────────────────────
print('=== VALIDATION BATCH (Ground Truth vs Predictions) ===')
val_labels = sorted(RUN_DIR.glob('val_batch*_labels.jpg'))[:2]
val_preds  = sorted(RUN_DIR.glob('val_batch*_pred.jpg'))[:2]
all_val    = []
all_titles = []
for lbl, pred in zip(val_labels, val_preds):
    all_val   += [str(lbl), str(pred)]
    all_titles += [lbl.name, pred.name]
show_images_row(all_val, all_titles, figsize=(22, 6))

In [ ]:
# ── args.yaml — xem lại config đã dùng ────────────────────────────────────────
args_path = RUN_DIR / 'args.yaml'
if args_path.exists():
    print('=== args.yaml ===')
    print(args_path.read_text())
else:
    print('Không tìm thấy args.yaml')

In [ ]:
# ── Tổng hợp tất cả file output ───────────────────────────────────────────────
print('=== TẤT CẢ FILE TRONG RUN DIR ===')
all_files = sorted(RUN_DIR.rglob('*'))
for f in all_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  {f.relative_to(RUN_DIR)}  ({size_kb:.1f} KB)')

In [ ]:
# ── Export ONNX + copy file về /kaggle/working/ ───────────────────────────────
best_model.export(format='onnx', imgsz=640, opset=12, simplify=True)

onnx_src = best_path.replace('.pt', '.onnx')
shutil.copy(best_path, '/kaggle/working/fire_best.pt')
shutil.copy(onnx_src,  '/kaggle/working/fire_best.onnx')

# Copy toàn bộ ảnh kết quả về working để dễ download
OUT_PLOTS = Path('/kaggle/working/fire_plots')
OUT_PLOTS.mkdir(exist_ok=True)

for ext in ['*.png', '*.jpg', '*.csv', '*.yaml']:
    for f in RUN_DIR.glob(ext):
        shutil.copy(f, OUT_PLOTS / f.name)

print('=== FILES SẴN SÀNG DOWNLOAD ===')
for f in sorted(OUT_PLOTS.iterdir()):
    print(f'  {f.name:<45} {f.stat().st_size/1024:.1f} KB')
print()
print('  fire_best.pt   → dùng trong testt_v2.py')
print('  fire_best.onnx → convert TensorRT')